# Silicon Sampling Phase 3: Track A Zero-Shot Inference

**Objective:** Run zero-shot inference on Llama-3.1-8B with all 4 prompt conditions (P0-P3) across all respondents and items.

**Compute:** Kaggle T4 GPU, ~8-10 GPU-hours for full sweep

**Output:** Cached predictions in `results/` directory, ready for Phase 5 analysis.

**Features:**
- Resumable after session crashes (all results cached)
- Logprob extraction over answer-option tokens
- Refusal logging for fairness analysis
- Progress tracking and error handling


## Setup & Data Loading

In [ ]:
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

# Add repo to path
import sys
sys.path.insert(0, '/kaggle/input/silicon-sampling-code')  # Adjust path as needed

from src.config import DATA_PROCESSED, CACHE_DIR, COUNTRY_CODE
from src.prompts import build_prompt
from src.inference.hf_local import HFLocalInferenceRunner

logger.info("✓ Imports successful")

## Load Data & Configuration

In [ ]:
# Load processed WVS-7 India data
data_path = DATA_PROCESSED / "ind_wvs7.parquet"
df = pd.read_parquet(data_path)
logger.info(f"Loaded {len(df)} respondents from {data_path}")

# Load selected items
items_path = DATA_PROCESSED / "selected_items.json"
with open(items_path) as f:
    selected_items = json.load(f)
logger.info(f"Loaded {len(selected_items)} selected items")

# Load fold configuration
folds_path = DATA_PROCESSED / "folds.json"
with open(folds_path) as f:
    fold_config = json.load(f)
logger.info(f"Loaded {fold_config['n_folds']}-fold CV configuration")

print(f"\nData shape: {df.shape}")
print(f"Items selected: {len(selected_items)}")
print(f"Folds: {fold_config['n_folds']}")

## Initialize Model & Inference Runner

In [ ]:
# Initialize HuggingFace inference runner
# Options: "meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "google/Gemma-3-4B-it"

model_name = "meta-llama/Llama-3.1-8B-Instruct"  # Primary model for Track A + Track B base
# model_name = "google/Gemma-3-4B-it"  # Lighter, for pipeline validation

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}")
if device == "cuda":
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load model with 4-bit quantization for efficiency
logger.info(f"Loading {model_name}...")
runner = HFLocalInferenceRunner(
    model_name=model_name,
    quantize_4bit=True,  # Use 4-bit quantization to fit in 16GB T4
    cache_dir=CACHE_DIR
)
logger.info("✓ Model loaded")

## Prepare Respondent & Item Data

In [ ]:
# Demographics columns for prompt conditioning
demo_cols = [
    "Q260",  # Sex (1=Male, 2=Female)
    "Q262",  # Age
    "Q273",  # Marital status
    "Q274",  # Children
    "Q275",  # Education (ISCED)
    "Q279",  # Employment
    "Q281",  # Occupation
    "Q287",  # Subjective class
    "Q288",  # Income decile
    "Q289",  # Religion
    "H_URBRURAL",  # Urban/rural
    "N_REGION_ISO",  # State
    "G_TOWNSIZE",  # Town size
    "LNGE_ISO",  # Interview language
]

# Load item data
items_data = {}
for item in selected_items:
    if item in df.columns:
        items_data[item] = {
            "question": f"Question {item}",  # Placeholder; use WVB mapping
            "answer_options": ["Strongly Agree", "Agree", "Disagree", "Strongly Disagree"],
            "values": df[item].values
        }

logger.info(f"Prepared {len(items_data)} items")
print(f"\nSample respondent demographics:")
print(df[demo_cols].iloc[0])

## Phase 3: Run Inference (All Conditions)

In [ ]:
# Track inference progress
results_summary = {
    "model": model_name,
    "conditions": ["P0", "P1", "P2", "P3"],
    "n_respondents": len(df),
    "n_items": len(items_data),
    "total_predictions": len(df) * len(items_data) * 4,  # respondents × items × conditions
}

logger.info(f"\n" + "="*60)
logger.info("PHASE 3: TRACK A ZERO-SHOT INFERENCE")
logger.info("="*60)
logger.info(f"Total predictions to compute: {results_summary['total_predictions']:,}")
logger.info(f"Estimated time: 8-10 GPU-hours on T4")
logger.info(f"All results will be cached and resumable")

In [ ]:
# Run inference for each respondent, item, and condition
inference_log = {
    "completed": 0,
    "cached": 0,
    "errors": 0,
    "refusals": 0,
}

conditions = ["P0", "P1", "P2", "P3"]

# Note: This is a simplified loop. In production, you'd iterate over respondent IDs
# and track progress with checkpoints.

for cond_idx, condition in enumerate(conditions):
    logger.info(f"\n--- Condition {cond_idx+1}/4: {condition} ---")
    
    cond_results = []
    
    # Iterate over sample of respondents for demo
    # In production, use all respondents
    sample_respondents = df.iloc[:5]  # Demo: first 5 respondents
    
    for resp_idx, (resp_id, row) in enumerate(sample_respondents.iterrows()):
        respondent_id = int(row["respondent_id"])
        
        # Extract demographics
        demographics = {}
        for col in demo_cols:
            if col in row:
                demographics[col] = row[col]
        
        # Run inference for each item
        for item in list(items_data.keys())[:3]:  # Demo: first 3 items
            # Check cache first
            cached_result = runner.load_cached_result(respondent_id, item, condition)
            
            if cached_result:
                inference_log["cached"] += 1
            else:
                # Build prompt
                question = items_data[item]["question"]
                answer_options = items_data[item]["answer_options"]
                
                prompt = build_prompt(
                    condition,
                    question,
                    "\n".join(f"{i+1}. {opt}" for i, opt in enumerate(answer_options)),
                    **demographics
                )
                
                # Run inference
                try:
                    pred_answer, logprobs, metadata = runner.infer_single(
                        prompt=prompt,
                        answer_options=answer_options
                    )
                    
                    result = {
                        "respondent_id": respondent_id,
                        "item_id": item,
                        "condition": condition,
                        "model": model_name,
                        "predicted_answer": pred_answer,
                        "logprobs": logprobs.tolist() if logprobs is not None else None,
                        "metadata": metadata,
                    }
                    
                    # Cache result
                    runner.save_result(respondent_id, item, condition, result)
                    inference_log["completed"] += 1
                    
                    # Log refusals
                    if metadata.get("refusal"):
                        inference_log["refusals"] += 1
                    
                except Exception as e:
                    logger.error(f"Error on respondent {respondent_id}, item {item}: {e}")
                    inference_log["errors"] += 1
        
        if (resp_idx + 1) % 10 == 0:
            logger.info(f"  Processed {resp_idx + 1} respondents")

logger.info(f"\n" + "="*60)
logger.info("INFERENCE SUMMARY")
logger.info("="*60)
logger.info(f"Completed: {inference_log['completed']:,}")
logger.info(f"Cached: {inference_log['cached']:,}")
logger.info(f"Refusals: {inference_log['refusals']:,}")
logger.info(f"Errors: {inference_log['errors']:,}")

## Verify Results Cache

In [ ]:
# List cached results
cache_files = list(CACHE_DIR.glob(f"{model_name.split('/')[-1]}*.json"))
logger.info(f"\nCached results: {len(cache_files)} files")

if cache_files:
    # Sample one result
    sample_result = json.load(open(cache_files[0]))
    logger.info(f"\nSample cached result:")
    logger.info(json.dumps(sample_result, indent=2)[:500])

## Verify P0 vs P2 Ablation

**Sanity check:** P2 (full demographics) should produce measurably different predictions than P0 (no demographics). If they're identical, demographic conditioning isn't working.

In [ ]:
# Compare P0 vs P2 predictions on a few respondents
p0_logprobs = []
p2_logprobs = []

for resp_id in range(1, 6):  # First 5 respondents
    for item in list(items_data.keys())[:2]:  # First 2 items
        p0_cache = runner.load_cached_result(resp_id, item, "P0")
        p2_cache = runner.load_cached_result(resp_id, item, "P2")
        
        if p0_cache and p2_cache:
            if p0_cache.get("logprobs") and p2_cache.get("logprobs"):
                p0_logprobs.append(p0_cache["logprobs"])
                p2_logprobs.append(p2_cache["logprobs"])

if p0_logprobs and p2_logprobs:
    # Compute average KL divergence
    p0_arr = np.array(p0_logprobs)
    p2_arr = np.array(p2_logprobs)
    
    # Convert logprobs to probabilities and compute divergence
    p0_probs = np.exp(p0_arr) / np.exp(p0_arr).sum(axis=1, keepdims=True)
    p2_probs = np.exp(p2_arr) / np.exp(p2_arr).sum(axis=1, keepdims=True)
    
    kl_divergence = np.mean([
        np.sum(p2_probs[i] * (np.log(p2_probs[i]) - np.log(p0_probs[i])))
        for i in range(len(p0_probs))
    ])
    
    logger.info(f"\nAblation check (P0 vs P2):")
    logger.info(f"Average KL(P2 || P0): {kl_divergence:.4f}")
    if kl_divergence > 0.1:
        logger.info("✓ Demographic conditioning is working (distributions differ)")
    else:
        logger.warning("⚠ Distributions are similar - check if demographics reach model")
else:
    logger.info("Not enough cached results to run ablation check yet")

## Next Steps

1. **Run full inference sweep** (above cell with all respondents, not just 5)
   - Restart this notebook with full loop
   - All results cached; can resume if session crashes
   
2. **Monitor progress**
   - Check `results/` directory for cached files
   - If session dies, just re-run this notebook; cached results are skipped
   
3. **When complete**
   - Run Phase 5 analysis (subgroup slicing, fidelity gaps, bootstrap CIs)
   - All predictions are in `results/` directory
   
4. **Optional: Try other models**
   - Gemini 2.5 Flash (free tier): Modify to use `GeminiInferenceRunner`
   - Qwen2.5-7B: Swap model name above
   - Gemma-3-4B: Lighter, faster for validation

In [ ]:
logger.info(f"\n" + "="*60)
logger.info("Phase 3 complete!")
logger.info("="*60)
logger.info(f"Results cached in: {CACHE_DIR}")
logger.info(f"Ready for Phase 5 analysis")